In [ ]:
pip install folium

In [13]:
import pandas as pd

# 1. Load data dari spreadsheet
df_direktori = pd.read_excel('prelist_SE2026.xlsx', sheet_name='prelist_SE2026')
df_keyword = pd.read_excel('prelist_SE2026.xlsx', sheet_name='keyword')

# 2. Ambil dan bersihkan data dari kolom 'keyword' (yang ingin dimasukkan) 
# dan kolom 'exclude' (yang ingin dikeluarkan)
keywords_include = df_keyword['keyword'].dropna().astype(str).str.strip()

# Pastikan nama kolom di excel sesuai, misalnya 'exclude'
keywords_exclude = df_keyword['exclude'].dropna().astype(str).str.strip()

# 3. Buat pola regex untuk yang di-include
pola_include = r'\b(' + '|'.join(keywords_include) + r')\b'

# 4. Buat kondisi filtering
# Kondisi 1: Harus mengandung kata dari kolom 'keyword'
kondisi_include = df_direktori['nama_usaha'].str.contains(pola_include, case=False, na=False, regex=True)

# Cek apakah daftar exclude ada isinya (mencegah error regex jika kolom exclude kosong)
if not keywords_exclude.empty:
    # Buat pola regex untuk yang di-exclude
    pola_exclude = r'\b(' + '|'.join(keywords_exclude) + r')\b'
    
    # Kondisi 2: Mengandung kata dari kolom 'exclude'
    kondisi_exclude = df_direktori['nama_usaha'].str.contains(pola_exclude, case=False, na=False, regex=True)
    
    # Gabungkan kondisi: True di include DAN ( & ) False di exclude (~ meniadakan kondisi)
    hasil_filter = df_direktori[kondisi_include & ~kondisi_exclude]
else:
    # Jika kolom exclude kosong, cukup filter dengan kondisi include saja
    hasil_filter = df_direktori[kondisi_include]

# 5. Simpan hasilnya
hasil_filter.to_excel('hasil_filter_usaha.xlsx', index=False)

C:\Users\mirfaiss\AppData\Local\Temp\ipykernel_35284\4132334323.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  kondisi_include = df_direktori['nama_usaha'].str.contains(pola_include, case=False, na=False, regex=True)
C:\Users\mirfaiss\AppData\Local\Temp\ipykernel_35284\4132334323.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  kondisi_exclude = df_direktori['nama_usaha'].str.contains(pola_exclude, case=False, na=False, regex=True)


In [14]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Baca file batas wilayah (GeoJSON)
print("Membaca data peta Pasaman...")
peta_pasaman = gpd.read_file('13.08_Pasaman.geojson')

# 2. Baca data daftar usaha (Excel)
print("Membaca data prelist usaha...")
df_usaha = pd.read_excel('hasil_filter_usaha.xlsx')

# Pastikan tidak ada data koordinat yang kosong (mencegah error)
df_usaha = df_usaha.dropna(subset=['latitude', 'longitude'])

# 3. Ubah koordinat Excel menjadi bentuk geometri (Titik/Point)
# Aturan koordinat spasial adalah (X, Y) yang berarti (Longitude, Latitude)
geometri_titik = [Point(lon, lat) for lon, lat in zip(df_usaha['longitude'], df_usaha['latitude'])]

# Buat GeoDataFrame (seperti DataFrame pandas, tapi punya kolom khusus untuk peta)
gdf_usaha = gpd.GeoDataFrame(df_usaha, geometry=geometri_titik)

# Samakan Sistem Referensi Koordinat (CRS) ke EPSG:4326 (Standar format GPS global)
gdf_usaha.set_crs(epsg=4326, inplace=True)
peta_pasaman.to_crs(epsg=4326, inplace=True)

# 4. Gabungkan seluruh poligon batas Pasaman menjadi satu kesatuan wilayah
# (Berjaga-jaga jika isi GeoJSON Anda terpecah-pecah per desa/kecamatan)
batas_wilayah_pasaman = peta_pasaman.geometry.unary_union

# 5. Pengecekan posisi: Apakah titik koordinat berada DI DALAM (within) batas Pasaman?
print("Melakukan pengecekan lokasi koordinat...")
# Ini akan menghasilkan nilai True jika di dalam, dan False jika di luar
mask_di_dalam = gdf_usaha.geometry.within(batas_wilayah_pasaman)

# 6. Pisahkan data berdasarkan hasil pengecekan
usaha_clean = df_usaha[mask_di_dalam]
usaha_false = df_usaha[~mask_di_dalam] # Simbol ~ membalikkan logika (yang False jadi True)

# 7. Simpan hasil ke Excel baru
# Kita hapus kolom 'geometry' sebelum disimpan karena Excel tidak mendukung format spasial
print("Menyimpan hasil ke Excel...")
usaha_clean.drop(columns='geometry', errors='ignore').to_excel('CLEAN_prelist_SE2026_ekraf.xlsx', index=False)
usaha_false.drop(columns='geometry', errors='ignore').to_excel('FALSE_prelist_SE2026_ekraf.xlsx', index=False)

print(f"\n--- Selesai! ---")
print(f"Data di DALAM Pasaman (CLEAN): {len(usaha_clean)} baris")
print(f"Data di LUAR Pasaman (FALSE): {len(usaha_false)} baris")

Membaca data peta Pasaman...
Membaca data prelist usaha...
Melakukan pengecekan lokasi koordinat...
Menyimpan hasil ke Excel...


C:\Users\mirfaiss\AppData\Local\Temp\ipykernel_35284\576274547.py:29: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  batas_wilayah_pasaman = peta_pasaman.geometry.unary_union



--- Selesai! ---
Data di DALAM Pasaman (CLEAN): 1413 baris
Data di LUAR Pasaman (FALSE): 48 baris


In [15]:
import pandas as pd
import json

# 1. Baca data Excel
file_excel = 'CLEAN_prelist_SE2026_ekraf.xlsx'
nama_sheet = 'Sheet1'

print("Membaca data Excel...")
df = pd.read_excel(file_excel)
df = df.dropna(subset=['latitude', 'longitude'])

# 2. Ubah data ke dalam format list of dictionaries
# Kita hanya mengambil kolom yang dibutuhkan agar ukuran file makin kecil
data_list = df[['nama_usaha', 'latitude', 'longitude']].to_dict(orient='records')

# 3. Bungkus data ke dalam format variabel JavaScript
# Ini trik agar data bisa dibaca langsung oleh HTML tanpa perlu web server lokal
js_content = f"var dataUsaha = {json.dumps(data_list)};"

# 4. Simpan ke file .js terpisah
with open('data_usaha.js', 'w') as f:
    f.write(js_content)

print(f"Berhasil! {len(data_list)} titik usaha telah diekstrak ke 'data_usaha.js'.")

Membaca data Excel...
Berhasil! 1413 titik usaha telah diekstrak ke 'data_usaha.js'.
